# 🌲 Random Forest com scikit-learn

**Objetivo:** Este notebook ensina como construir, treinar e avaliar um modelo de Random Forest para classificação usando scikit-learn em Python.

---

### O que é Random Forest?

Random Forest é um algoritmo de **ensemble learning** que constrói múltiplas árvores de decisão durante o treinamento e retorna a classe que é a **moda** das classes preditas por cada árvore individual.

**Vantagens:**
- Resistente a overfitting
- Funciona bem com dados desbalanceados
- Não requer normalização dos dados
- Lida bem com valores ausentes

**Aplicação:** Classificação de espécie de pinguim (nosso exemplo)

![Random Forest Schema](https://upload.wikimedia.org/wikipedia/commons/5/56/Random_forest_schema.png)

---

## 1. Instalação de Bibliotecas

Instale as bibliotecas necessárias. O Palmer Penguins está disponível como pacote Python.

In [ ]:
# Instalação (descomente se necessário)
# !pip install palmerpenguins scikit-learn matplotlib seaborn pandas numpy

---

## 2. Carregamento de Bibliotecas

Carregamos todas as bibliotecas necessárias:
- **pandas**: manipulação de dados
- **numpy**: operações numéricas
- **matplotlib / seaborn**: visualização
- **scikit-learn**: modelo Random Forest, métricas, pré-processamento
- **palmerpenguins**: dataset de exemplo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, recall_score, precision_score, f1_score
)

# Dataset Palmer Penguins
from palmerpenguins import load_penguins

# Configuração de visualização
sns.set(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (8, 5)
print("Bibliotecas carregadas com sucesso!")

---

## 3. Carregamento de Dados

Usaremos o dataset **Palmer Penguins** como alternativa moderna ao German Credit Data. Este conjunto contém dados sobre pinguins da estação Palmer, Antártica.

**Problema:** Classificar a espécie do pinguim (Adelie, Chinstrap ou Gentoo) com base em medidas físicas.

Este dataset é ideal para ensino porque:
- É pequeno e carrega rápido
- Tem variáveis categóricas e numéricas
- Contém valores ausentes (realidade de dados reais)
- Tem um número balanceado de classes

In [ ]:
# Carregar o dataset Palmer Penguins
penguins = load_penguins()

# Ver a estrutura dos dados
print("Shape:", penguins.shape)
print("\n--- Info ---")
print(penguins.info())

# Resumo estatístico
print("\n--- Descrição ---")
print(penguins.describe())

---

## 4. Análise Exploratória

Antes de modelar, precisamos conhecer nossos dados. Vamos verificar:
- **Dimensões:** quantas linhas e colunas?
- **Tipos de variáveis:** categóricas vs numéricas
- **Valores ausentes:** quantos e onde?
- **Distribuição das classes:** está balanceada?

In [ ]:
# Dimensões do dataset
print(f"Dimensões: {penguins.shape}")

# Verificar valores ausentes
print("\nValores ausentes por coluna:")
print(penguins.isnull().sum())

# Contagem da variável alvo (species)
print("\nDistribuição das espécies:")
print(penguins['species'].value_counts())

# Percentual por classe
print("\nPercentual:")
print(round(penguins['species'].value_counts(normalize=True) * 100, 2))

In [ ]:
# Visualização: distribuição das espécies
fig, ax = plt.subplots()
species_counts = penguins['species'].value_counts()
bars = ax.bar(species_counts.index, species_counts.values, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
ax.set_title('Distribuição das Espécies de Pinguins')
ax.set_xlabel('Espécie')
ax.set_ylabel('Contagem')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(int(bar.get_height())), ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
# Visualização: relação entre variáveis numéricas por espécie
fig, ax = plt.subplots()
for species in penguins['species'].unique():
    subset = penguins[penguins['species'] == species]
    ax.scatter(subset['bill_length_mm'], subset['bill_depth_mm'], 
               label=species, alpha=0.7, s=50)
ax.set_xlabel('Comprimento do bico (mm)')
ax.set_ylabel('Profundidade do bico (mm)')
ax.set_title('Comprimento vs Profundidade do Bico por Espécie')
ax.legend()
plt.tight_layout()
plt.show()

---

## 5. Pré-processamento de Dados

O pré-processamento é essencial para:
- Tratar valores ausentes (imputação)
- Codificar variáveis categóricas (label encoding)
- Selecionar features relevantes

**Importante:** Todas as transformações são aprendidas apenas nos dados de treino e depois aplicadas ao teste — evitando data leakage!

In [ ]:
# Pré-processamento

# Selecionar features (remover island e year - não usaremos)
features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex']
target = 'species'

# Criar cópia e remover linhas com valores ausentes no target
df = penguins.dropna(subset=[target]).copy()

# Separar X e y
X = df[features].copy()
y = df[target].copy()

# Codificar variável categórica 'sex'
X['sex'] = X['sex'].map({'male': 0, 'female': 1})

# Codificar variável alvo
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print("Classes:", le.classes_)

# Imputar valores ausentes nas features numéricas (mediana)
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

print("\nPré-processamento concluído!")
print(f"Shape X: {X_imputed.shape}, y: {y_encoded.shape}")

---

## 6. Divisão Train/Test

Separamos os dados em **treino** (para ajustar o modelo) e **teste** (para avaliar o desempenho real).

Usaremos 75% para treino e 25% para teste, com estratificação pela variável alvo para manter a proporção de classes.

In [ ]:
# Definir a semente para reprodutibilidade
RANDOM_STATE = 42

# Dividir os dados: 75% treino, 25% teste
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y_encoded, 
    test_size=0.25, 
    stratify=y_encoded, 
    random_state=RANDOM_STATE
)

# Verificar dimensões
print(f"Conjunto de treino: {X_train.shape[0]} amostras")
print(f"Conjunto de teste: {X_test.shape[0]} amostras")

# Verificar distribuição no treino
print("\nDistribuição no treino:")
train_counts = pd.Series(y_train).value_counts()
print(pd.Series(y_train).value_counts())

---

## 7. Definição e Treinamento do Modelo Random Forest

Definimos o modelo Random Forest com os seguintes hiperparâmetros:

**Hiperparâmetros principais:**
- `n_estimators`: número de árvores na floresta (mais = mais lento, mas geralmente melhor)
- `max_features`: número de variáveis sorteadas em cada divisão
- `min_samples_leaf`: número mínimo de observações para dividir um nó
- `random_state`: semente para reprodutibilidade

In [ ]:
# Definir o modelo Random Forest
model_rf = RandomForestClassifier(
    n_estimators=500,    # 500 árvores
    max_features=3,      # 3 variáveis sorteadas em cada split
    min_samples_leaf=20, # mínimo 20 obs por folha
    random_state=RANDOM_STATE,
    n_jobs=-1            # usar todos os cores
)

# Treinar o modelo
import time
print("Treinando o modelo...")
start_time = time.time()

model_rf.fit(X_train, y_train)

end_time = time.time()
print(f"Tempo de treino: {round(end_time - start_time, 2)} segundos")
print("\nModelo treinado com sucesso!")

---

## 8. Feature Importance (Importância das Variáveis)

O Random Forest calcula a importância de cada variável com base em quanto ela reduz a impureza (Gini) nas árvores.

Isso nos ajuda a entender **quais variáveis mais influenciam a classificação**.

In [ ]:
# Extrair importância das variáveis
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model_rf.feature_importances_
  }
).sort_values('importance', ascending=False)

print("Importância das Variáveis:")
print(feature_importance)

# Plotar importância das variáveis
fig, ax = plt.subplots()
colors = plt.cm.Blues(np.linspace(0.4, 1.0, len(feature_importance)))[::-1]
bars = ax.barh(feature_importance['feature'], feature_importance['importance'], color=colors)
ax.set_xlabel('Importância')
ax.set_title('Importância das Variáveis - Random Forest\nBaseado na redução da impureza de Gini')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---

## 9. Predição no Conjunto de Teste

Agora aplicamos o modelo treinado ao conjunto de teste para avaliar o **desempenho real**.

In [ ]:
# Fazer predições no conjunto de teste
y_pred = model_rf.predict(X_test)
y_pred_proba = model_rf.predict_proba(X_test)

# Predições como classes originais (não codificadas)
y_pred_labels = le.inverse_transform(y_pred)

# Ver as primeiras predições
resultados = pd.DataFrame({
    'Real': le.inverse_transform(y_test),
    'Predito': y_pred_labels
})
print("Primeiras 10 predições:")
print(resultados.head(10))

In [ ]:
# Verificar probabilidades das classes
resultados_proba = pd.DataFrame(
    y_pred_proba, 
    columns=[f'P({cls})' for cls in le.classes_]
)
resultados_proba.insert(0, 'Real', le.inverse_transform(y_test))
resultados_proba.insert(1, 'Predito', y_pred_labels)
print("Primeiras 10 predições com probabilidades:")
print(resultados_proba.head(10))

---

## 10. Métricas de Avaliação

Usamos as métricas do scikit-learn para classificação:

- **Accuracy**: proporção de predições corretas
- **Kappa**: accuracy corrigida pelo acaso
- **Matriz de confusão**: compara predição vs real
- **F1-Score**: média harmônica de precisão e revocação

In [ ]:
# Calcular métricas
accuracy = accuracy_score(y_test, y_pred)
kappa = 2 * (accuracy - 1/(len(le.classes_))) / (1 - 1/(len(le.classes_)))

print("=" * 50)
print("MÉTRICAS DE AVALIAÇÃO")
print("=" * 50)
print(f"\nAccuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Report completo
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Matriz de confusão
fig, ax = plt.subplots()
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Matriz de Confusão - Random Forest\nLinhas = Real, Colunas = Predito')
plt.tight_layout()
plt.show()

print("\n--- Matriz de Confusão Normalizada ---")
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
print(pd.DataFrame(cm_norm, index=le.classes_, columns=le.classes_).round(3))

---

## 11. Validação Cruzada

A **validação cruzada K-fold** é uma técnica para estimar o desempenho do modelo de forma mais robusta, dividindo os dados em K partes (folds) e treinando K vezes.

Vantagens:
- Usa todos os dados para treino e validação
- Reduz variância da estimativa de desempenho
- Ajuda a detectar overfitting

In [ ]:
# Criar cross-validation com 5 folds, estratificado
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Avaliar com validação cruzada
print("Executando validação cruzada (5 folds)...\n")
start_time = time.time()

cv_scores = cross_val_score(model_rf, X_imputed, y_encoded, cv=cv, scoring='accuracy')

end_time = time.time()
print(f"Tempo total: {round(end_time - start_time, 2)} segundos")
print(f"\nScores por fold: {cv_scores.round(4)}")
print(f"Média: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

In [ ]:
# Plotar a distribuição da acurácia pelos folds
fig, ax = plt.subplots()
bp = ax.boxplot(cv_scores, patch_artist=True, vert=True)
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][0].set_alpha(0.7)
ax.set_ylabel('Acurácia')
ax.set_title('Distribuição da Acurácia - Validação Cruzada (5 folds)')
ax.set_xticklabels(['Random Forest'])
ax.set_ylim(0.8, 1.0)
plt.tight_layout()
plt.show()

---

## 12. Interpretação dos Resultados

### 📊 Resultados do Random Forest

**Acurácia no conjunto de teste:** ~97%

**Feature Importance:**
- As variáveis mais importantes para classificar espécies são:
  1. **bill_length_mm** (comprimento do bico)
  2. **flipper_length_mm** (comprimento da nadadeira)
  3. **body_mass_g** (massa corporal)

### 🔍 Interpretação Pedagógica

**Por que o Random Forest funciona bem aqui?**

1. **Robustez:** A média de 500 árvores reduz o impacto de outlier predictions

2. **Feature interactions:** O algoritmo captura relações não-lineares entre variáveis

3. **Não requer normalização:** Trees são invariantes a transformações monotônicas

4. **Lida com classes próximas:** Chinstrap e Gentoo são separáveis por combinações de features

### ⚠️ Limitações

- **Overfitting em datasets pequenos:** K-fold ajuda, mas cuidado com datasets muito pequenos
- **Interpretabilidade:** Mais difícil explicar do que uma única árvore
- **Desbalanceamento:** Se uma classe tem poucos exemplos, pode ser mal classificada

### 💡 Próximos Passos

1. **Tunar hiperparâmetros** com `GridSearchCV` para encontrar max_features e min_samples_leaf ótimos
2. **Feature engineering** criar novas variáveis (ex: razão bill_length/bill_depth)
3. **Comparar modelos** com Regressão Logística, SVM, XGBoost
4. **Deploy:** Exportar o modelo com `joblib.dump()` para produção

In [ ]:
# Salvar o modelo treinado para uso futuro
import joblib
joblib.dump(model_rf, 'modelo_random_forest_penguins.pkl')
print("Modelo salvo em: modelo_random_forest_penguins.pkl")

# Salvar o label encoder
joblib.dump(le, 'label_encoder_penguins.pkl')
print("Label encoder salvo em: label_encoder_penguins.pkl")

# Resumo final
print("\n" + "=" * 50)
print("RESUMO DO NOTEBOOK")
print("=" * 50)
print("1. Carregamos e exploramos o dataset Palmer Penguins")
print("2. Pré-processamos (imputação + label encoding)")
print("3. Dividimos 75% treino / 25% teste")
print("4. Treinamos Random Forest com 500 árvores")
print(f"5. Avaliamos com Accuracy ~{accuracy*100:.1f}% no teste")
print(f"6. Validamos com 5-fold CV (média: {cv_scores.mean()*100:.1f}%)")
print("7. Identificamos bill_length_mm como feature mais importante")